In [1]:
import numpy as np
from torch import nn, optim
from torch.utils.data import DataLoader, TensorDataset
import torch
from matplotlib import pyplot as plt
from jordanutils import *
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)
import pandas as pd
from collections import defaultdict

rng = np.random.default_rng(seed=123)

In [2]:
MODES = ['random', 'int', 'upper', 'lower', 'ortho', 'random_eps']

In [3]:
def setup_device():
    try:
        import torch_directml

        device = torch_directml.device()
        backend = "directml"    
    except ImportError:
        if torch.cuda.is_available():
            device = torch.device("cuda")
            backend = "cuda"
        else:
            device = torch.device("cpu")
            backend = "cpu"
    return device


## Training on all "signle" datasets 

In [10]:
# ---- Define the PyTorch model ----
class SimpleNN(nn.Module):
    def __init__(self, d):
        super(SimpleNN, self).__init__()
        self.flatten = nn.Flatten()
        self.layers = nn.Sequential(
            nn.Linear(d * d, 256),
            nn.ReLU(),
            nn.Linear(256, 256),
            nn.ReLU(),
            nn.Linear(256, 256),
            nn.ReLU(),
            nn.Linear(256, 256),
            nn.ReLU(),
            nn.Linear(256, 256),
            nn.ReLU(),
            nn.Linear(256, 256),
            nn.ReLU(),
            nn.Linear(256, 256),
            nn.ReLU(),
            nn.Linear(256, 256),
            nn.ReLU(),
            nn.Linear(256, d),
        )

    def forward(self, x):
        x = self.flatten(x)
        x = self.layers(x)
        return x

# ---- Training + Evaluation function ----
def generate_and_train_model(X, y, device, verbose=1, epochs=50):
    # --- Dataset setup ---
    d = X[0].shape[0]
    
    X_train, X_val, y_train, y_val = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=21
    )

    # --- Convert to tensors ---
    X_train = torch.tensor(X_train, dtype=torch.float32)
    y_train = torch.tensor(y_train, dtype=torch.long)
    X_val = torch.tensor(X_val, dtype=torch.float32)
    y_val = torch.tensor(y_val, dtype=torch.long)

    # --- DataLoaders ---
    train_dataset = TensorDataset(X_train, y_train)
    val_dataset = TensorDataset(X_val, y_val)
    train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=128, shuffle=False)

    # --- Model, Loss, Optimizer ---
    model = SimpleNN(d).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=1e-3)

    # --- Early Stopping setup ---
    best_val_acc = 0.0
    patience = 5
    counter = 0
    best_weights = None

    for epoch in range(epochs):
        model.train()
        # Initialize running loss for the epoch
        running_loss = 0.0
        counter = 0
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            preds = model(xb)
            loss = criterion(preds, yb)
            loss.backward()
            optimizer.step()
            # Accumulate training loss
            running_loss += loss.item() * xb.size(0)
            counter += yb.size(0)

        # Calculate average training loss for the epoch
        train_loss = running_loss / counter

        # --- Validation Loop ---
        model.eval()
        correct, total = 0, 0
        # Initialize running validation loss
        val_running_loss = 0.0
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(device), yb.to(device)
                preds = model(xb)
                # Calculate validation loss
                val_loss = criterion(preds, yb)
                val_running_loss += val_loss.item() * xb.size(0)

                pred_labels = preds.argmax(1)
                correct += (pred_labels == yb).sum().item()
                total += yb.size(0)
        
        # Calculate average validation loss for the epoch
        avg_val_loss = val_running_loss / total
        val_acc = correct / total

        if verbose:
            print(f"Epoch {epoch+1:02d} - Train Loss: {train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | Val Acc: {val_acc:.4f}")
        
        # --- Early Stopping ---
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            counter = 0
            best_weights = model.state_dict()
        else:
            counter += 1
            if epoch == 10:
                counter = 0
            if counter >= patience and epoch > 10:
                if verbose:
                    print(f"Early stopping at epoch {epoch+1}")
                break

    # --- Restore best model ---
    if best_weights is not None:
        model.load_state_dict(best_weights)

    return model



In [5]:
device = setup_device()
size_per_class = 50000
d = 5
# X, y = generate_testset(d, size_per_class=size_per_class, mode='random')
# model = generate_and_train_model(X, y, device)

In [6]:
testsets = {}
for mode in MODES:
    _test_mode, _eps_test = ("random", 0.01) if mode == 'random_eps' else (mode, None)
    testsets[mode] = generate_testset(d, size_per_class=1000, mode=_test_mode, eps=_eps_test)

In [12]:
def run_testcase(model, X_test):    
    X_test = torch.tensor(X_test, dtype=torch.float32).to(device)
    model.eval()

    with torch.no_grad():
        outputs = model(X_test)
        y_predicted = torch.argmax(outputs, dim=1).cpu().numpy()

    return y_predicted

In [14]:
def run_and_gather_results(testsets):
    train_modes = MODES
    test_modes = MODES
    full_results = defaultdict(dict)
    accuracy_results = defaultdict(dict)
    for train_mode in train_modes: 
        _train_mode, _eps_train = ("random", 0.01) if train_mode == 'random_eps' else (train_mode, None)
        X_train, y_train = generate_testset(d, 50_000, _train_mode, _eps_train)
        model = generate_and_train_model(X_train, y_train, device, epochs=50)
        for test_mode in test_modes:
            X_test, y_test = testsets[test_mode]
            y_pred = run_testcase(model, X_test)
            accuracy = accuracy_score(y_test, y_pred)
            cm = confusion_matrix(y_test, y_pred)

            full_results[test_mode][train_mode] = cm
            accuracy_results[test_mode][train_mode] = accuracy

            print(f" ###### Train: {train_mode} | Test: {test_mode} | Accuracy: {accuracy:.4f} ######")

    return full_results, accuracy_results

In [15]:
full_results, accuracy_results = run_and_gather_results(testsets)

Epoch 01 - Train Loss: 0.9448 | Val Loss: 0.8803 | Val Acc: 0.5705
Epoch 02 - Train Loss: 0.7741 | Val Loss: 0.7426 | Val Acc: 0.6406
Epoch 03 - Train Loss: 0.7295 | Val Loss: 0.7462 | Val Acc: 0.6345
Epoch 04 - Train Loss: 0.6955 | Val Loss: 0.7031 | Val Acc: 0.6598
Epoch 05 - Train Loss: 0.6800 | Val Loss: 0.6772 | Val Acc: 0.6746
Epoch 06 - Train Loss: 0.6530 | Val Loss: 0.6615 | Val Acc: 0.6814
Epoch 07 - Train Loss: 0.6352 | Val Loss: 0.6457 | Val Acc: 0.6916
Epoch 08 - Train Loss: 0.6236 | Val Loss: 0.6431 | Val Acc: 0.6936
Epoch 09 - Train Loss: 0.6124 | Val Loss: 0.6339 | Val Acc: 0.6958
Epoch 10 - Train Loss: 0.6001 | Val Loss: 0.6285 | Val Acc: 0.7034
Epoch 11 - Train Loss: 0.5897 | Val Loss: 0.6223 | Val Acc: 0.7057
Epoch 12 - Train Loss: 0.5834 | Val Loss: 0.6294 | Val Acc: 0.7027
Early stopping at epoch 12
 ###### Train: random | Test: random | Accuracy: 0.7034 ######
 ###### Train: random | Test: int | Accuracy: 0.7046 ######
 ###### Train: random | Test: upper | Accuracy

In [13]:
def save_data(data_to_save, filename):
    flat_data = {}
    for outer_key, inner_dict in data_to_save.items():
        for inner_key, array in inner_dict.items():
            flat_data[f"test:{outer_key}/train:{inner_key}"] = array

    # 2. Save the flattened data
    np.savez_compressed(f'{filename}.npz', **flat_data)
    print(f"Data saved to {filename}.npz")

In [17]:
save_data(accuracy_results, 'accuracy_results')
save_data(full_results, 'full_results')

Data saved to accuracy_results.npz
Data saved to full_results.npz


In [18]:
print(accuracy_results)

defaultdict(<class 'dict'>, {'random': {'random': 0.7034, 'int': 0.6974, 'upper': 0.3726, 'lower': 0.3932, 'ortho': 0.4026, 'random_eps': 0.7}, 'int': {'random': 0.7046, 'int': 0.692, 'upper': 0.3792, 'lower': 0.404, 'ortho': 0.4024, 'random_eps': 0.699}, 'upper': {'random': 0.557, 'int': 0.5856, 'upper': 0.972, 'lower': 0.8274, 'ortho': 0.4376, 'random_eps': 0.5538}, 'lower': {'random': 0.6506, 'int': 0.6244, 'upper': 0.857, 'lower': 0.9852, 'ortho': 0.4056, 'random_eps': 0.6492}, 'ortho': {'random': 0.5274, 'int': 0.4806, 'upper': 0.3872, 'lower': 0.4062, 'ortho': 1.0, 'random_eps': 0.4004}, 'random_eps': {'random': 0.6662, 'int': 0.6846, 'upper': 0.3648, 'lower': 0.3, 'ortho': 0.3462, 'random_eps': 0.6942}})


### Training on `full` dataset

In [15]:
def run_full_training(testsets):
    X_list = []
    y_list = []
    for mode in MODES:
        _mode, _eps = ("random", 0.01) if mode == 'random_eps' else (mode, None)
        X, y = generate_testset(5, 50_000, mode=_mode, eps=_eps)
        X_list.append(X)
        y_list.append(y)
    X_train = np.concat(X_list, axis=0)
    y_train = np.concat(y_list, axis=0)
    model = generate_and_train_model(X_train, y_train, device, epochs=50)
    accuracy_results = defaultdict(dict)
    full_results = defaultdict(dict)
    for test_mode in MODES:
        X_test, y_test = testsets[test_mode]
        y_pred = run_testcase(model, X_test)
        accuracy = accuracy_score(y_test, y_pred)
        cm = confusion_matrix(y_test, y_pred)

        full_results[test_mode]["full"] = cm
        accuracy_results[test_mode]["full"] = accuracy

        print(f" ###### Train: full | Test: {test_mode} | Accuracy: {accuracy:.4f} ######")

    return full_results, accuracy_results

In [16]:
full_results, accuracy_results = run_full_training(testsets)

save_data(accuracy_results, 'accuracy_results')
save_data(full_results, 'full_results')

Epoch 01 - Train Loss: 0.6176 | Val Loss: 0.5027 | Val Acc: 0.7677
Epoch 02 - Train Loss: 0.4762 | Val Loss: 0.4396 | Val Acc: 0.7965
Epoch 03 - Train Loss: 0.4342 | Val Loss: 0.4257 | Val Acc: 0.8017
Epoch 04 - Train Loss: 0.4082 | Val Loss: 0.3919 | Val Acc: 0.8171
Epoch 05 - Train Loss: 0.3907 | Val Loss: 0.4134 | Val Acc: 0.8108
Epoch 06 - Train Loss: 0.3881 | Val Loss: 0.3818 | Val Acc: 0.8226
Epoch 07 - Train Loss: 0.3940 | Val Loss: 0.3705 | Val Acc: 0.8275
Epoch 08 - Train Loss: 0.3802 | Val Loss: 0.3781 | Val Acc: 0.8253
Epoch 09 - Train Loss: 0.3676 | Val Loss: 0.3642 | Val Acc: 0.8321
Epoch 10 - Train Loss: 0.3734 | Val Loss: 0.3645 | Val Acc: 0.8313
Epoch 11 - Train Loss: 0.3564 | Val Loss: 0.3610 | Val Acc: 0.8332
Epoch 12 - Train Loss: 0.3524 | Val Loss: 0.3510 | Val Acc: 0.8376
Epoch 13 - Train Loss: 0.3526 | Val Loss: 0.3874 | Val Acc: 0.8195
Early stopping at epoch 13
 ###### Train: full | Test: random | Accuracy: 0.6866 ######
 ###### Train: full | Test: int | Accurac